# Day 2 — Очистка текста и подготовка признаков

Цель: получить стабильную функцию предобработки, применимую и к обучающим данным, и к новым текстам на inference.

Что делается:
- очистка: URL (включая `t.co/...` без схемы), `@mentions`, HTML, пунктуация, лишние пробелы;
- нормализация: Unicode NFKC, нижний регистр, `ё → е`, сжатие повторов символов;
- фильтрация пустых/сломанных текстов;
- метки: нижний регистр + числовой `label_id`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from classification.config import CLEAN_DATA_PATH, LABEL_COL, RAW_DATA_PATH, TEXT_COL
from classification.load_data import load_dataset
from classification.preprocess import (
    clean_text,
    encode_labels,
    is_valid_text,
    preprocess_dataframe,
)

raw_df = load_dataset(RAW_DATA_PATH)
clean_df = preprocess_dataframe(raw_df)
print(f"RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"CLEAN_DATA_PATH: {CLEAN_DATA_PATH}")
print(f"Строк до: {len(raw_df)}, после: {len(clean_df)}")

RAW_DATA_PATH: C:\Users\Aleksandra\PycharmProjects\classification\data\raw\rusentitweet.csv
CLEAN_DATA_PATH: C:\Users\Aleksandra\PycharmProjects\classification\data\processed\clean.csv
Строк до: 8842, после: 8839


## 1. Ручная проверка `clean_text`

In [2]:
manual = pd.DataFrame(
    {
        "raw": [
            "@axstr0 Соболезную https://t.co/pM9npo9dUd",
            "@marygeneralova В моём доме только одно животное https://t.co/e0988EdFnB",
            "@AyupovRustam Вааа 😍😍😍",
            "Помимо Алины, у меня ещё радость...\r\n\r\nГЕМОГЛОБИН ПОВЫСИЛСЯ",
            "<b>Новость</b> тут: www.example.com/news",
            None,
        ]
    }
)
manual["clean"] = manual["raw"].map(clean_text)
manual["valid"] = manual["clean"].map(is_valid_text)
display(manual)

,raw,clean,valid
0,@axstr0 Соболезную https://t.co/pM9npo9dUd,соболезную,True
1,@marygeneralova В моём доме только одно животн...,в моем доме только одно животное,True
2,@AyupovRustam Вааа 😍😍😍,ваа,True
3,"Помимо Алины, у меня ещё радость...\r\n\r\nГЕМ...",помимо алины у меня еще радость гемоглобин пов...,True
4,<b>Новость</b> тут: www.example.com/news,новость тут,True
5,NaN,,False


## 2. Примеры из датасета до и после

In [3]:
sample = (
    raw_df[raw_df[TEXT_COL].str.contains("@", regex=False)]
    .head(8)[[TEXT_COL, LABEL_COL]]
    .reset_index(drop=True)
)
sample["clean"] = sample[TEXT_COL].map(clean_text)
display(sample)

,text,label,clean
0,@namaskaramsaroo Мотоцикль,neutral,мотоцикль
1,@buybread_ я не с порядке!!!!,negative,я не с порядке
2,@AyupovRustam Вааа 😍😍😍,neutral,ваа
3,"@well_syxjker77 да, кинь телегу пж",neutral,да кинь телегу пж
4,@ange1flyhigh В следующий раз буду до победног...,positive,в следующий раз буду до победного ее закрывать...
5,@ru_rbc доллар по 30 сделать не хотят значит,neutral,доллар по 30 сделать не хотят значит
6,"@LimitaVIP Удивительный гiмн...\nУдивительно, ...",negative,удивительный гiмн удивительно что пока еще не ...
7,@volhelmut а если не нравиться?,neutral,а если не нравиться


## 3. Пустые и сломанные примеры

In [4]:
cleaned_raw = raw_df[TEXT_COL].map(clean_text)
removed = raw_df.loc[~cleaned_raw.map(is_valid_text)].copy()
removed["clean"] = cleaned_raw.loc[removed.index]
print(f"Удалено: {len(removed)}")
display(removed[[TEXT_COL, "clean", LABEL_COL]])

Удалено: 3


,text,clean,label
1420,в https://t.co/UNKGpGL2aV,в,neutral
1815,@xionnccya Я,я,neutral
6899,@shutupgrese я 🖤,я,positive


## 4. Нормализация и кодирование меток

In [5]:
label_example = pd.DataFrame(
    {
        TEXT_COL: ["хорошо", "нормально", "плохо", "другое"],
        LABEL_COL: [" POSITIVE ", "neutral", "Negative", "sport"],
    }
)
encoded, mapping = encode_labels(label_example)
display(encoded)
print("mapping:", mapping)
print("distribution in clean.csv:")
display(
    pd.DataFrame(
        {"count": clean_df[LABEL_COL].value_counts(), "code": clean_df.groupby(LABEL_COL)["label_id"].first()}
    )
)

,text,label,label_id
0,хорошо,positive,2
1,нормально,neutral,1
2,плохо,negative,0
3,другое,sport,3


mapping: {'negative': 0, 'neutral': 1, 'positive': 2, 'sport': 3}
distribution in clean.csv:


,count,code
label,,
negative,2638,0
neutral,4271,1
positive,1930,2


## 5. Проверка качества очистки

In [6]:
checks = {
    "пустые тексты": int((clean_df[TEXT_COL] == "").sum()),
    "короткие/без букв": int((~clean_df[TEXT_COL].map(is_valid_text)).sum()),
    "остатки @mentions": int(clean_df[TEXT_COL].str.contains("@", regex=False).sum()),
    "остатки t.co": int(clean_df[TEXT_COL].str.contains("t.co", regex=False).sum()),
    "остатки http/www": int(clean_df[TEXT_COL].str.contains(r"http|www\.", regex=True).sum()),
    "остатки HTML": int(clean_df[TEXT_COL].str.contains(r"<[^>]+>", regex=True).sum()),
    "пропуски label_id": int(clean_df["label_id"].isna().sum()),
}
display(pd.Series(checks, name="value").to_frame())

,value
пустые тексты,0
короткие/без букв,0
остатки @mentions,0
остатки t.co,0
остатки http/www,0
остатки HTML,0
пропуски label_id,0


## Выводы

- `clean_text` один и тот же используется для обучающих данных и inference, его можно импортировать из `classification.preprocess`.
- На реальных твитах очистка убирает `@mentions`, `https://t.co/...` и `t.co/...` без схемы, эмодзи/пунктуацию, применяет Unicode NFKC и `ё → е`, сжимает повторы (`ваааа → ваа`).
- Из 8842 строк удалены 3: после очистки остались односимвольные тексты (`в`, `я`).
- Метки нормализуются в нижний регистр и получают стабильный `label_id`: `negative=0`, `neutral=1`, `positive=2`. Для других датасетов кодирование работает по отсортированным оставшимся классам.
- Проверки после очистки: пустых текстов и остатков URL/mentions/HTML нет.